# **Introduction and Data Description**

## Introduction

In this competition your task is to predict whether a passenger was transported to an alternate dimension during the Spaceship Titanic's collision with the spacetime anomaly. To help you make these predictions, you're given a set of personal records recovered from the ship's damaged computer system.

## Data Description

**train.csv - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.**

* PassengerId - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
* HomePlanet - The planet the passenger departed from, typically their planet of permanent residence.
* CryoSleep - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
* Cabin - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
Destination - The planet the passenger will be debarking to.
* Age - The age of the passenger.
* VIP - Whether the passenger has paid for special VIP service during the voyage.
* RoomService, FoodCourt, ShoppingMall, Spa, VRDeck - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
* Name - The first and last names of the passenger.
* Transported - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.

**test.csv - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.**

**sample_submission.csv - A submission file in the correct format.**

* PassengerId - Id for each passenger in the test set.
* Transported - The target. For each passenger, predict either True or False.

# **Import Libraries and Data**

## Import Libraries

In [404]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from catboost import CatBoostClassifier

from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import cross_val_score, train_test_split

## Import Data

In [405]:
train=pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test=pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
train.shape,test.shape

((8693, 14), (4277, 13))

In [406]:
train.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True


In [407]:
test.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers


# **Data Cleaning**

### Missing Values

In [408]:
train['Transported'].isna().sum()

0

In [409]:
data=pd.concat([train,test],axis=0,ignore_index=True)
data = pd.concat([data, data['Cabin'].str.split('/', expand=True)], axis=1)
data.rename(columns={0: 'Deck', 1: 'Num', 2: 'Side'}, inplace=True)
data.drop(columns=['PassengerId','Name','Cabin','Num'],inplace=True)
data.isna().sum()/data.shape[0]

HomePlanet      0.022205
CryoSleep       0.023901
Destination     0.021126
Age             0.020817
VIP             0.022822
RoomService     0.020278
FoodCourt       0.022282
ShoppingMall    0.023593
Spa             0.021897
VRDeck          0.020663
Transported     0.329761
Deck            0.023053
Side            0.023053
dtype: float64

In [410]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12970 entries, 0 to 12969
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    12682 non-null  object 
 1   CryoSleep     12660 non-null  object 
 2   Destination   12696 non-null  object 
 3   Age           12700 non-null  float64
 4   VIP           12674 non-null  object 
 5   RoomService   12707 non-null  float64
 6   FoodCourt     12681 non-null  float64
 7   ShoppingMall  12664 non-null  float64
 8   Spa           12686 non-null  float64
 9   VRDeck        12702 non-null  float64
 10  Transported   8693 non-null   object 
 11  Deck          12671 non-null  object 
 12  Side          12671 non-null  object 
dtypes: float64(6), object(7)
memory usage: 1.3+ MB


In [411]:
for col in data.columns:
    if data[col].dtype==object:
        print(col,'\b:')
        print(data[col].value_counts(dropna=False))

HomePlanet:
Earth     6865
Europa    3133
Mars      2684
NaN        288
Name: HomePlanet, dtype: int64
CryoSleep:
False    8079
True     4581
NaN       310
Name: CryoSleep, dtype: int64
Destination:
TRAPPIST-1e      8871
55 Cancri e      2641
PSO J318.5-22    1184
NaN               274
Name: Destination, dtype: int64
VIP:
False    12401
NaN        296
True       273
Name: VIP, dtype: int64
Transported:
True     4378
False    4315
NaN      4277
Name: Transported, dtype: int64
Deck:
F      4239
G      3781
E      1323
B      1141
C      1102
D       720
A       354
NaN     299
T        11
Name: Deck, dtype: int64
Side:
S      6381
P      6290
NaN     299
Name: Side, dtype: int64


In [412]:
data['HomePlanet'].fillna('Earth',inplace=True)
data['CryoSleep'].fillna(False,inplace=True)
data['Destination'].fillna('TRAPPIST-1e',inplace=True)
data['VIP'].fillna(False,inplace=True)
data['Deck'].fillna('F',inplace=True)
data['Side'].fillna(method='ffill',inplace=True)
data['Transported'].fillna(method='ffill',inplace=True)

for col in data.columns:
    if data[col].dtype!=object:
        data[col].fillna(data[col].median(),inplace=True)
        
data.isna().sum()[data.isna().sum()>0]

Series([], dtype: int64)

### Standadizing

In [413]:
for col in data.columns:
    if data[col].dtype=='float64':
        print(col,'\b:',data[col].var())

Age: 202.74791719986092
RoomService: 411863.8565454137
FoodCourt: 2458743.4090403705
ShoppingMall: 341235.4634393278
Spa: 1251594.3696245237
VRDeck: 1365756.0028085478


In [414]:
data1=data.copy()
for col in data1.columns:
    if data1[col].dtype=='float64':
        data1[col]=(data1[col]-data1[col].mean())/data1[col].std()
        print(data1[col].var())

0.9999999999999999
1.0000000000000002
1.0000000000000002
1.0
1.0000000000000002
0.9999999999999999


### OneHot Encoding

In [415]:
data1['Transported'].astype(int)
data1=pd.get_dummies(data1)
data1.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,False,0.720904,False,-0.340277,-0.281811,-0.292354,-0.269697,-0.257091,False,0,...,0,1,0,0,0,0,0,0,1,0
1,False,-0.332544,False,-0.170433,-0.276072,-0.249557,0.221031,-0.219440,True,1,...,0,0,0,0,0,1,0,0,0,1
2,False,2.055271,True,-0.273274,1.998745,-0.292354,5.732555,-0.215162,False,0,...,1,0,0,0,0,0,0,0,0,1
3,False,0.299525,False,-0.340277,0.536409,0.342753,2.705954,-0.091943,False,0,...,1,0,0,0,0,0,0,0,0,1
4,False,-0.894383,False,0.131858,-0.237170,-0.033860,0.235333,-0.255379,True,1,...,0,0,0,0,0,1,0,0,0,1


### Data Split

In [416]:
train1=data1.iloc[:train.shape[0],:]
test1=data1.iloc[train.shape[0]:,:]
train1.shape,test1.shape

((8693, 25), (4277, 25))

### Duplicated Values

In [417]:
train1.duplicated().sum()

1938

In [418]:
train1.drop_duplicates(inplace=True)
train1.duplicated().sum()

0

### Outliers

In [419]:
clf = LocalOutlierFactor(contamination = 0.01)
outliers = clf.fit_predict(train1)
train2 = train1[np.where(outliers == 1, True, False)]
train2.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,False,0.720904,False,-0.340277,-0.281811,-0.292354,-0.269697,-0.257091,False,0,...,0,1,0,0,0,0,0,0,1,0
1,False,-0.332544,False,-0.170433,-0.276072,-0.249557,0.221031,-0.219440,True,1,...,0,0,0,0,0,1,0,0,0,1
2,False,2.055271,True,-0.273274,1.998745,-0.292354,5.732555,-0.215162,False,0,...,1,0,0,0,0,0,0,0,0,1
3,False,0.299525,False,-0.340277,0.536409,0.342753,2.705954,-0.091943,False,0,...,1,0,0,0,0,0,0,0,0,1
4,False,-0.894383,False,0.131858,-0.237170,-0.033860,0.235333,-0.255379,True,1,...,0,0,0,0,0,1,0,0,0,1


# **Model Development**

### Split Data

In [420]:
train2=pd.concat([train2,train2],axis=0,ignore_index=True)
x=train2.drop(columns='Transported')
pred=test1.drop(columns='Transported')
y=train2['Transported']
x.shape,y.shape,pred.shape

((13374, 24), (13374,), (4277, 24))

In [421]:
xtrain,xtest,ytrain,ytest= train_test_split(x, y, test_size = 0.5)
xtrain.shape,ytrain.shape,xtest.shape,ytest.shape

((6687, 24), (6687,), (6687, 24), (6687,))

### Modeling

In [422]:
cat=CatBoostClassifier(eval_metric= "Accuracy")
model=cat.fit(xtrain,ytrain,eval_set=(xtest,ytest),verbose=500)

Learning rate set to 0.050669
0:	learn: 0.7517571	test: 0.7421863	best: 0.7421863 (0)	total: 3.82ms	remaining: 3.81s
500:	learn: 0.8863466	test: 0.8175565	best: 0.8177060 (496)	total: 1.96s	remaining: 1.96s
999:	learn: 0.9261253	test: 0.8343054	best: 0.8343054 (996)	total: 3.9s	remaining: 0us

bestTest = 0.8343053686
bestIteration = 996

Shrink model to first 997 iterations.


# **Submission**

In [423]:
result=model.predict(pred)

In [424]:
submission=pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')
submission.head(2)

,PassengerId,Transported
0,0013_01,False
1,0018_01,False


In [425]:
submission['Transported']=result
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [426]:
submission.to_csv('/kaggle/working/submission.csv',index=False)